# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the metadata to list all record sets and their fields. All references use the `@id`.

In [ ]:
# List record sets and their fields by @id

# Get the list of record sets
record_sets = metadata.record_set

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        if hasattr(rs, 'field') and rs.field:
            for field in rs.field:
                print(f"  Field @id: {field.id} | name: {field.name} | dataType: {getattr(field, 'data_type', 'N/A')}")
        elif hasattr(rs, 'column') and rs.column:
            # Some RecordSets may use 'column' instead of 'field'
            for col in rs.column:
                print(f"  Column @id: {col.id} | name: {col.name} | dataType: {getattr(col, 'data_type', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All access and references are made using the `@id` values shown above.

If available, extract multiple record sets.

In [ ]:
# Build a list of record set @ids
record_set_ids = []
rs_objects = []
if metadata.record_set:
    for rs in metadata.record_set:
        record_set_ids.append(rs.id)
        rs_objects.append(rs)

dataframes = {}

# Extract all records for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for record set @id: {record_set_id} has columns: {df.columns.tolist()}")
    print(df.head(2))

# Choose one record set for detailed analysis
chosen_record_set_id = record_set_ids[0] if record_set_ids else None
if chosen_record_set_id:
    print(f"Using record set @id: {chosen_record_set_id} for analysis.")
    print(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All columns and fields referenced by their `@id`.

In [ ]:
# EDA on selected record set
df = dataframes.get(chosen_record_set_id)
if df is not None:
    # Find numeric fields by inspecting column names and dtypes
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    
    print("Numeric fields available:", numeric_fields)
    # Choose a numeric field for analysis (e.g. first numeric field found)
    numeric_field_id = numeric_fields[0] if numeric_fields else None
    
    if numeric_field_id:
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Grouping by a categorical field
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
        group_field_id = group_fields[0] if group_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} showing mean {numeric_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Using matplotlib for simple plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of field @id: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by group field @id: {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored the FAIR^2 dataset using Croissant metadata and accessed all entities by their unique `@id`.
- Loaded metadata and record sets, inspected available fields and columns.
- Performed EDA on numeric and categorical fields. Identified potential outliers and normalized field distributions.
- Visualized field distributions and group variations.
- This dataset provides insights into predictors of knowledge adoption in rangeland management and can inform policy, gender inclusion, and adaptation strategies in Northern Kenya.

For full documentation, see [mlcroissant documentation](https://mlcroissant.org/) and FAIR^2 dataset documentation.